# ATDL Phase 4 — Controlled Class-Imbalance Generation

This phase creates mild, moderate, and severe training-only long-tail variants from the Phase 2 train split. It samples at image level and retains every bounding box on every selected image. Validation and test remain in the original processed folders and are referenced unchanged by each dataset YAML. No training occurs here.

## 1. Locate project inputs and define class-frequency schedules

In [1]:
from pathlib import Path
import json, math, random, shutil
from collections import Counter
import pandas as pd
import yaml

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "requirements-local.txt").is_file() and (p / "data").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open the atdl_final_project folder in VS Code.")
DATA_ROOT = PROJECT_ROOT / "data"
PROCESSED = DATA_ROOT / "processed"
VARIANTS_ROOT = DATA_ROOT / "imbalance_variants"
RESULTS = PROJECT_ROOT / "results"
PHASE2_SUMMARY = RESULTS / "phase2" / "phase2_dataset_summary.json"
CONVERSION_MANIFEST = RESULTS / "phase2" / "yolo_conversion_manifest.csv"
CLASS_NAMES = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]
SEED = 42
FORCE_REBUILD = False
RETENTION = {
    "mild":     [1.00, 0.90, 0.80, 0.70, 0.60, 0.50],
    "moderate": [1.00, 0.80, 0.60, 0.45, 0.30, 0.20],
    "severe":   [1.00, 0.65, 0.40, 0.25, 0.15, 0.08],
}
if not PHASE2_SUMMARY.is_file() or not CONVERSION_MANIFEST.is_file():
    raise FileNotFoundError("Run Phase 2 first.")
phase2 = json.loads(PHASE2_SUMMARY.read_text(encoding="utf-8"))
if phase2.get("validation_status") != "passed":
    raise ValueError("Phase 2 summary is not passed.")
conversion = pd.read_csv(CONVERSION_MANIFEST)
train = conversion[conversion["split"] == "train"].copy()
if train.empty:
    raise ValueError("No Phase 2 training images were found.")
print("Phase 2 training images:", len(train))
print("Validation/test sources remain:", phase2.get("split_image_counts"))


Phase 2 training images: 1257
Validation/test sources remain: {'train': 1257, 'test': 276, 'val': 266}


## 2. Read image-level class presence and object frequencies

In [2]:
def read_label(label_path):
    rows = []
    for line in Path(label_path).read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) != 5:
            raise ValueError(f"Bad YOLO row in {label_path}: {line}")
        class_id = int(parts[0])
        if not 0 <= class_id < len(CLASS_NAMES):
            raise ValueError(f"Class ID out of range in {label_path}")
        rows.append((class_id, [float(v) for v in parts[1:]]))
    return rows

records = []
image_presence = Counter()
object_counts = Counter()
for row in train.itertuples(index=False):
    labels = read_label(row.yolo_label)
    present = sorted({class_id for class_id, _ in labels})
    for class_id in present:
        image_presence[class_id] += 1
    for class_id, _ in labels:
        object_counts[class_id] += 1
    records.append({
        "image": Path(row.yolo_image), "label": Path(row.yolo_label),
        "classes": present, "objects": len(labels),
    })

frequency = pd.DataFrame([{
    "class_id": i, "class_name": CLASS_NAMES[i],
    "training_images": image_presence[i], "objects": object_counts[i],
} for i in range(len(CLASS_NAMES))]).sort_values("training_images", ascending=False)
display(frequency)
class_order = frequency["class_id"].tolist()


,class_id,class_name,training_images,objects
1,1,inclusion,267,687
2,2,patches,239,611
3,3,pitted_surface,211,307
0,0,crazing,210,483
4,4,rolled-in_scale,210,443
5,5,scratches,210,384


## 3. Select whole images to meet target class-presence counts

In [3]:
def select_whole_images(records, targets, seed):
    rng = random.Random(seed)
    remaining = list(records)
    rng.shuffle(remaining)
    deficits = dict(targets)
    selected = []
    while any(value > 0 for value in deficits.values()):
        eligible = []
        for record in remaining:
            useful = [cid for cid in record["classes"] if deficits.get(cid, 0) > 0]
            if useful:
                score = sum(min(deficits[cid] / max(targets[cid], 1), 1.0) for cid in useful)
                eligible.append((score, record, useful))
        if not eligible:
            break
        best_score = max(item[0] for item in eligible)
        top = [item for item in eligible if item[0] == best_score]
        _, chosen, _ = rng.choice(top)
        selected.append(chosen)
        remaining.remove(chosen)
        for cid in chosen["classes"]:
            if cid in deficits and deficits[cid] > 0:
                deficits[cid] -= 1
    return selected, deficits

max_presence = max(image_presence.values())
targets_by_level = {}
for level, ratios in RETENTION.items():
    targets = {}
    for rank, class_id in enumerate(class_order):
        targets[class_id] = max(1, math.ceil(max_presence * ratios[rank]))
    targets_by_level[level] = targets

print("Reference max image presence:", max_presence)
for level, targets in targets_by_level.items():
    print(level, {CLASS_NAMES[cid]: count for cid, count in targets.items()})


Reference max image presence: 267
mild {'inclusion': 267, 'patches': 241, 'pitted_surface': 214, 'crazing': 187, 'rolled-in_scale': 161, 'scratches': 134}
moderate {'inclusion': 267, 'patches': 214, 'pitted_surface': 161, 'crazing': 121, 'rolled-in_scale': 81, 'scratches': 54}
severe {'inclusion': 267, 'patches': 174, 'pitted_surface': 107, 'crazing': 67, 'rolled-in_scale': 41, 'scratches': 22}


## 4. Materialize train-only variants and save manifests

In [4]:
VARIANTS_ROOT.mkdir(parents=True, exist_ok=True)
distribution_rows = []
names_yaml = {idx: name for idx, name in enumerate(CLASS_NAMES)}
val_images = (PROCESSED / "images" / "val").resolve()
test_images = (PROCESSED / "images" / "test").resolve()

for level, targets in targets_by_level.items():
    variant_root = VARIANTS_ROOT / level
    manifest_path = variant_root / "selection_manifest.csv"
    complete = manifest_path.is_file()
    if complete:
        saved = pd.read_csv(manifest_path)
        complete = len(saved) > 0 and all(Path(p).is_file() for p in saved["image"]) and all(
            Path(p).is_file() for p in saved["label"]
        )
    if complete and not FORCE_REBUILD:
        selected_frame = pd.read_csv(manifest_path)
        print(f"Reusing completed {level} variant ({len(selected_frame)} images).")
    else:
        if variant_root.exists() and any(variant_root.iterdir()) and not FORCE_REBUILD:
            raise FileExistsError(f"Partial variant found at {variant_root}; inspect it or set FORCE_REBUILD=True.")
        selected, remaining_deficits = select_whole_images(records, targets, SEED)
        if remaining_deficits and any(remaining_deficits.values()):
            print("Unmet image-presence targets due to multi-class image selection:", remaining_deficits)
        rows = []
        for record in selected:
            image_out = variant_root / "images" / "train" / record["image"].name
            label_out = variant_root / "labels" / "train" / record["label"].name
            image_out.parent.mkdir(parents=True, exist_ok=True)
            label_out.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(record["image"], image_out)
            shutil.copy2(record["label"], label_out)
            rows.append({
                "image": str(image_out), "label": str(label_out),
                "source_image": str(record["image"]), "source_label": str(record["label"]),
                "classes_present": ",".join(CLASS_NAMES[cid] for cid in record["classes"]),
                "object_count": record["objects"], "split": "train",
            })
        selected_frame = pd.DataFrame(rows)
        manifest_path.parent.mkdir(parents=True, exist_ok=True)
        selected_frame.to_csv(manifest_path, index=False)

    yaml_data = {
        "path": str(variant_root.resolve()),
        "train": "images/train",
        "val": str(val_images),
        "test": str(test_images),
        "names": names_yaml,
    }
    (variant_root / "data.yaml").write_text(yaml.safe_dump(yaml_data, sort_keys=False), encoding="utf-8")

    selected_image_counts, selected_object_counts = Counter(), Counter()
    for label_path in selected_frame["label"]:
        rows = read_label(label_path)
        for cid in {class_id for class_id, _ in rows}:
            selected_image_counts[cid] += 1
        for cid, _ in rows:
            selected_object_counts[cid] += 1
    for cid, name in enumerate(CLASS_NAMES):
        distribution_rows.append({
            "variant": level, "class_name": name, "target_train_images": targets[cid],
            "actual_train_images": selected_image_counts[cid],
            "actual_train_objects": selected_object_counts[cid],
            "selected_total_images": len(selected_frame),
        })

distribution = pd.DataFrame(distribution_rows)
distribution_path = RESULTS / "phase4_final_distribution_report.csv"
distribution.to_csv(distribution_path, index=False)
summary = {
    "source_split": "Phase 2 train only", "validation_and_test_unchanged": True,
    "seed": SEED, "retention_schedules": RETENTION,
    "variants": {level: {
        "train_images": int(distribution[distribution.variant == level]["selected_total_images"].iloc[0]),
        "data_yaml": str((VARIANTS_ROOT / level / "data.yaml").resolve()),
        "manifest": str((VARIANTS_ROOT / level / "selection_manifest.csv").resolve()),
    } for level in RETENTION},
    "distribution_report": str(distribution_path.resolve()),
}
(RESULTS / "phase4_summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
display(distribution)
print("Saved Phase 4 distribution:", distribution_path)


Unmet image-presence targets due to multi-class image selection: {1: 0, 2: 2, 3: 3, 0: 0, 4: 0, 5: 0}


,variant,class_name,target_train_images,actual_train_images,actual_train_objects,selected_total_images
0,mild,crazing,187,187,426,1109
1,mild,inclusion,267,267,687,1109
2,mild,patches,241,239,611,1109
3,mild,pitted_surface,214,211,307,1109
4,mild,rolled-in_scale,161,161,340,1109
5,mild,scratches,134,134,248,1109
6,moderate,crazing,121,121,275,808
7,moderate,inclusion,267,267,687,808
8,moderate,patches,214,214,553,808
9,moderate,pitted_surface,161,161,233,808


Saved Phase 4 distribution: ./results/phase4_final_distribution_report.csv


## 5. Verify variants use only training images and preserve labels

In [5]:
source_train = {str(Path(p).resolve()) for p in train["yolo_image"]}
for level in RETENTION:
    manifest = pd.read_csv(VARIANTS_ROOT / level / "selection_manifest.csv")
    leaked = [p for p in manifest["source_image"] if str(Path(p).resolve()) not in source_train]
    if leaked:
        raise ValueError(f"Non-train images entered {level}: {leaked[:3]}")
    if len(manifest) != manifest["source_image"].nunique():
        raise ValueError(f"Duplicate source image in {level}")
    for row in manifest.itertuples(index=False):
        source_rows = read_label(row.source_label)
        copied_rows = read_label(row.label)
        if source_rows != copied_rows:
            raise ValueError(f"Annotations changed during copy for {row.source_image}")
    print(f"{level}: {len(manifest)} selected source images; all label rows preserved.")
print("Original validation and test folders were not modified.")


mild: 1109 selected source images; all label rows preserved.
moderate: 808 selected source images; all label rows preserved.
severe: 597 selected source images; all label rows preserved.
Original validation and test folders were not modified.


## Phase 4 outputs

Variants: data/imbalance_variants/mild, moderate, severe  
Distribution report: results/phase4_final_distribution_report.csv  
This phase deliberately creates experimental imbalance; report actual distributions rather than calling the original dataset imbalanced.